#  GenAI Project | Intelligent Log Classification System using DeepSeek R1, BERT, NLP & Regex


# Data Load

In [27]:
import pandas as pd

In [28]:
df= pd.read_csv("dataset/synthetic_logs.csv")
df.head(5)

,timestamp,source,log_message,target_label,complexity
0,6/27/2025 7:20,ModernCRM,nova.osapi_compute.wsgi.server [req-b9718cd8-f...,HTTP Status,bert
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error,bert
2,1/17/2025 1:29,AnalyticsEngine,Unauthorized access to data was attempted,Security Alert,bert
3,7/12/2025 0:24,ModernHR,nova.osapi_compute.wsgi.server [req-4895c258-b...,HTTP Status,bert
4,6/2/2025 18:25,BillingSystem,nova.osapi_compute.wsgi.server [req-ee8bc8ba-9...,HTTP Status,bert


In [29]:
df.target_label.unique()


<StringArray>
[        'HTTP Status',      'Critical Error',      'Security Alert',
               'Error', 'System Notification',      'Resource Usage',
         'User Action',      'Workflow Error', 'Deprecation Warning']
Length: 9, dtype: str

In [30]:
df.complexity.unique()

<StringArray>
['bert', 'regex', 'llm']
Length: 3, dtype: str

In [31]:
from sentence_transformers import SentenceTransformer

#Loading pretrained sentence transformer model
model= SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1764.32it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [32]:
# Generate embeddings for log messages
embeddings = model.encode(df['log_message'].tolist())

In [33]:
embeddings[:6]

array([[-0.10293962,  0.03354593, -0.02202606, ...,  0.00457791,
        -0.04259717,  0.00322621],
       [ 0.00804573, -0.03573923,  0.04938737, ...,  0.01538319,
        -0.06230948, -0.02774663],
       [-0.00908223,  0.13003924, -0.05275566, ...,  0.02014104,
        -0.05117098, -0.02930296],
       [-0.09751043,  0.04911299, -0.03977422, ...,  0.024775  ,
        -0.03546078, -0.000186  ],
       [-0.10468339,  0.05926036, -0.02488496, ...,  0.02502052,
        -0.03719298, -0.02568911],
       [-0.09372505,  0.04970007, -0.05461299, ...,  0.02133468,
        -0.03557463, -0.02681496]], shape=(6, 384), dtype=float32)

# Clustering

In [34]:
# perform DB scan clustering
from sklearn.cluster import DBSCAN


In [35]:
dbscan= DBSCAN(eps=0.2, min_samples=1 , metric= 'cosine')
clusters= dbscan.fit_predict(embeddings)

In [36]:
# adding cluster labels to the dataframes
df['cluster']= clusters

In [37]:
# checking the clusters
df.head(3)

,timestamp,source,log_message,target_label,complexity,cluster
0,6/27/2025 7:20,ModernCRM,nova.osapi_compute.wsgi.server [req-b9718cd8-f...,HTTP Status,bert,0
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error,bert,1
2,1/17/2025 1:29,AnalyticsEngine,Unauthorized access to data was attempted,Security Alert,bert,2


In [38]:
df[df.cluster==2]

,timestamp,source,log_message,target_label,complexity,cluster
2,1/17/2025 1:29,AnalyticsEngine,Unauthorized access to data was attempted,Security Alert,bert,2
1410,8/7/2025 18:51,ModernCRM,An unusual data access attempt was detected,Security Alert,bert,2
1426,3/3/2025 16:50,ThirdPartyAPI,Identified a possible unauthorized data access...,Security Alert,bert,2


In [39]:
#Count how many logs are in each cluster
cluster_counts = df['cluster'].value_counts()

In [40]:
#Keep only clusters with more than 10 logs
large_clusters = cluster_counts[cluster_counts>10].index

In [41]:
for cluster in large_clusters:
    print(f"cluster{cluster}:")
    print(df[df['cluster']== cluster]['log_message'].head(5).to_string(index=False)) #Printing first 5 log messages from each cluster

cluster0:
nova.osapi_compute.wsgi.server [req-b9718cd8-f6...
nova.osapi_compute.wsgi.server [req-4895c258-b2...
nova.osapi_compute.wsgi.server [req-ee8bc8ba-92...
nova.osapi_compute.wsgi.server [req-f0bffbc3-5a...
nova.osapi_compute.wsgi.server [req-2bf7cfee-a2...
cluster5:
nova.compute.claims [req-a07ac654-8e81-416d-bfb...
nova.compute.claims [req-d6986b54-3735-4a42-907...
nova.compute.claims [req-72b4858f-049e-49e1-b31...
nova.compute.claims [req-5c8f52bd-8e3c-41f0-95a...
nova.compute.claims [req-d38f479d-9bb9-4276-968...
cluster11:
User User685 logged out.
 User User395 logged in.
 User User225 logged in.
User User494 logged out.
 User User900 logged in.
cluster13:
Backup started at 2025-05-14 07:06:55.
Backup started at 2025-02-15 20:00:19.
  Backup ended at 2025-08-08 13:06:23.
Backup started at 2025-11-14 08:27:43.
Backup started at 2025-12-09 10:19:11.
cluster7:
Multiple bad login attempts detected on user 85...
Multiple login failures occurred on user 9052 a...
  User 7153 made

# Classification Stage 1: Regex


In [42]:
import re
def classify_with_regex(log_message):
    regex_patterns = {
        r"User User\d+ logged (in|out).": "User Action",
        r"Backup (started|ended) at .*": "System Notification",
        r"Backup completed successfully.": "System Notification",
        r"System updated to version .*": "System Notification",
        r"File .* uploaded successfully by user .*": "System Notification",
        r"Disk cleanup completed successfully.": "System Notification",
        r"System reboot initiated by user .*": "System Notification",
        r"Account with ID .* created by .*": "User Action"
    }
    for pattern, label in regex_patterns.items():
        if re.search(pattern, log_message, re.IGNORECASE):
            return label
    return None

In [43]:
classify_with_regex("User User395 logged in.")

'User Action'

In [44]:
df['regex_label'] = df['log_message'].apply(classify_with_regex)
df

,timestamp,source,log_message,target_label,complexity,cluster,regex_label
0,6/27/2025 7:20,ModernCRM,nova.osapi_compute.wsgi.server [req-b9718cd8-f...,HTTP Status,bert,0,NaN
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error,bert,1,NaN
2,1/17/2025 1:29,AnalyticsEngine,Unauthorized access to data was attempted,Security Alert,bert,2,NaN
3,7/12/2025 0:24,ModernHR,nova.osapi_compute.wsgi.server [req-4895c258-b...,HTTP Status,bert,0,NaN
4,6/2/2025 18:25,BillingSystem,nova.osapi_compute.wsgi.server [req-ee8bc8ba-9...,HTTP Status,bert,0,NaN
...,...,...,...,...,...,...,...
2405,8/13/2025 7:29,ModernHR,nova.osapi_compute.wsgi.server [req-96c3ec98-2...,HTTP Status,bert,0,NaN
2406,1/11/2025 5:32,ModernHR,User 3844 account experienced multiple failed ...,Security Alert,bert,7,NaN
2407,8/3/2025 3:07,ThirdPartyAPI,nova.metadata.wsgi.server [req-b6d4a270-accb-4...,HTTP Status,bert,0,NaN
2408,11/11/2025 11:52,BillingSystem,Email service affected by failed transmission,Critical Error,bert,1,NaN


In [45]:
df[df.regex_label.notnull()]

,timestamp,source,log_message,target_label,complexity,cluster,regex_label
7,10/11/2025 8:44,ModernHR,File data_6169.csv uploaded successfully by us...,System Notification,regex,4,System Notification
14,1/4/2025 1:43,ThirdPartyAPI,File data_3847.csv uploaded successfully by us...,System Notification,regex,4,System Notification
15,5/1/2025 9:41,ModernCRM,Backup completed successfully.,System Notification,regex,8,System Notification
18,2/22/2025 17:49,ModernCRM,Account with ID 5351 created by User634.,User Action,regex,9,User Action
27,9/24/2025 19:57,ThirdPartyAPI,User User685 logged out.,User Action,regex,11,User Action
...,...,...,...,...,...,...,...
2376,6/27/2025 8:47,ModernCRM,System updated to version 2.0.5.,System Notification,regex,21,System Notification
2381,9/5/2025 6:39,ThirdPartyAPI,Disk cleanup completed successfully.,System Notification,regex,32,System Notification
2394,4/3/2025 13:13,ModernHR,Disk cleanup completed successfully.,System Notification,regex,32,System Notification
2395,5/2/2025 14:29,ThirdPartyAPI,Backup ended at 2025-05-06 11:23:16.,System Notification,regex,13,System Notification


#so only total of 500 rows were classified by out regex classifier.. now the remaining rows which were not classified by the regex, we will put in seperate col to work on it.
#Clustering helps discover patterns in unlabeled logs, and regex is then used to efficiently classify similar logs in real-time. Together, they enable both pattern discovery and fast execution.
#The idea is, because of the clustering we could find the patterns, and then due to regex we could classify them, by using manual selection..

In [46]:
df_non_regex = df[df['regex_label'].isnull()].copy()
df_non_regex

,timestamp,source,log_message,target_label,complexity,cluster,regex_label
0,6/27/2025 7:20,ModernCRM,nova.osapi_compute.wsgi.server [req-b9718cd8-f...,HTTP Status,bert,0,NaN
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error,bert,1,NaN
2,1/17/2025 1:29,AnalyticsEngine,Unauthorized access to data was attempted,Security Alert,bert,2,NaN
3,7/12/2025 0:24,ModernHR,nova.osapi_compute.wsgi.server [req-4895c258-b...,HTTP Status,bert,0,NaN
4,6/2/2025 18:25,BillingSystem,nova.osapi_compute.wsgi.server [req-ee8bc8ba-9...,HTTP Status,bert,0,NaN
...,...,...,...,...,...,...,...
2405,8/13/2025 7:29,ModernHR,nova.osapi_compute.wsgi.server [req-96c3ec98-2...,HTTP Status,bert,0,NaN
2406,1/11/2025 5:32,ModernHR,User 3844 account experienced multiple failed ...,Security Alert,bert,7,NaN
2407,8/3/2025 3:07,ThirdPartyAPI,nova.metadata.wsgi.server [req-b6d4a270-accb-4...,HTTP Status,bert,0,NaN
2408,11/11/2025 11:52,BillingSystem,Email service affected by failed transmission,Critical Error,bert,1,NaN


# Classification Stage 2 : Classification Using Embeddings

In [47]:
print(
    df_non_regex['target_label'].value_counts()
    [df_non_regex['target_label'].value_counts()<=5].index.tolist()
)

['Workflow Error', 'Deprecation Warning']


In [48]:
#Now we are working on the rest of the data
df_non_legacy = df_non_regex[df_non_regex.source!='LegacyCRM']
df_non_legacy.source.unique()

<StringArray>
['ModernCRM', 'AnalyticsEngine', 'ModernHR', 'BillingSystem', 'ThirdPartyAPI']
Length: 5, dtype: str

In [49]:
#embeddings + a classifier to automatically identify which system a log message belongs to, based on its text.
#generate embeddings for log messages
filtered_embeddings=model.encode((df_non_legacy['log_message'].tolist()))
filtered_embeddings[:2]

array([[-1.02939621e-01,  3.35459262e-02, -2.20260601e-02,
         1.55105593e-03, -9.86920949e-03, -1.78956255e-01,
        -6.34409785e-02, -6.01761676e-02,  2.81108841e-02,
         5.99620454e-02, -1.72618646e-02,  1.43367471e-03,
        -1.49560004e-01,  3.15285777e-03, -5.66031076e-02,
         2.71685384e-02, -1.49890380e-02, -3.54037546e-02,
        -3.62936258e-02, -1.45410234e-02, -5.61493542e-03,
         8.75539109e-02,  4.55120429e-02,  2.50963680e-02,
         1.00187706e-02,  1.24267014e-02, -1.39923617e-01,
         7.68696442e-02,  3.14095207e-02, -4.15248703e-03,
         4.36902680e-02,  1.71250310e-02, -8.00951496e-02,
         5.74005917e-02,  1.89091805e-02,  8.55262280e-02,
         3.96399237e-02, -1.34371832e-01, -1.44364929e-03,
         3.06707853e-03,  1.76854029e-01,  4.44886740e-03,
        -1.69275180e-02,  2.24266723e-02, -4.35049348e-02,
         6.09028572e-03, -9.98169184e-03, -6.23973049e-02,
         1.07372776e-02, -6.04895549e-03, -7.14660808e-0

In [50]:
# Now training the model for prediction using logistic regression
# This code trains a model to automatically classify log messages into the correct system (target_label) and checks how accurate it is
X = filtered_embeddings
y = df_non_legacy['target_label']
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
X_train , X_test , y_train , y_test = train_test_split(X, y, test_size = 0.3, random_state= 42)
clf = LogisticRegression(max_iter= 1000)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
report = classification_report(y_test, y_pred)
print(report)

# CONCLUSION:
#model is highly accurate (99%) at classifying log messages
#Embeddings + Logistic Regression are working extremely well

                precision    recall  f1-score   support

Critical Error       0.91      1.00      0.95        48
         Error       0.98      0.89      0.93        47
   HTTP Status       1.00      1.00      1.00       304
Resource Usage       1.00      1.00      1.00        49
Security Alert       1.00      0.99      1.00       123

      accuracy                           0.99       571
     macro avg       0.98      0.98      0.98       571
  weighted avg       0.99      0.99      0.99       571



In [51]:
#storing our trained model (clf) into a file on disk
#joblib.dump(...) saves your trained machine learning model so you can reuse it later without retraining
import joblib
joblib.dump(clf, '../models/log_classifier.joblib')
#we can use this for Predictions and inferences

['../models/log_classifier.joblib']